# ShadowWeave — World Model Training

Trains a U-Net world model on MuJoCo synthetic rollouts.  
Works locally (MPS/CPU) or on Colab (T4/L4 CUDA).

**Steps:**
1. Setup — install deps, detect device
2. Generate rollout data from MuJoCo sim
3. Train WorldModel (U-Net, ~31M params)
4. Evaluate IOU at each prediction horizon
5. Download checkpoint (Colab) or it's saved locally

In [1]:
# ── Cell 1: Environment setup ────────────────────────────────────────
import os, sys

# Prevent CUDA allocator fragmentation (safe on all versions)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.system('apt-get install -y -q libgl1-mesa-glx libegl1-mesa libosmesa6 2>/dev/null')
    os.environ['MUJOCO_GL'] = 'egl'
    os.system('pip install -q mujoco omegaconf tqdm')
else:
    pass

import math, time, pathlib, textwrap
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm  # auto-selects notebook widget vs plain tty bar

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'Colab: {IN_COLAB}')
print(f'PyTorch: {torch.__version__}')

Device: cuda
Colab: True
PyTorch: 2.11.0+cu128


In [2]:
# ── Cell 2: Config (all hyperparams here, zero magic numbers below) ──
from types import SimpleNamespace

CFG = SimpleNamespace(
    # sim
    room_size       = 6.0,
    camera_height   = 1.6,
    camera_fov      = 90,
    sim_fps         = 30,
    # depth map resolution
    depth_h         = 128,
    depth_w         = 128,
    # shadow raycaster
    num_rays        = 512,
    grid_cells      = 9,
    # data generation
    steps_per_ep    = 60,         # usable timesteps per episode
    n_train_eps     = 80,         # increase for better model (200 ideal)
    n_val_eps       = 20,
    prediction_horizons = [1, 3, 5, 10],  # seconds ahead
    # world model
    base_channels   = 32,         # 32 for MPS/CPU (~7.8M params, ~4x faster); Colab gets 64
    batch_size      = 16,         # 16 fits M3 Pro MPS; halves batches/epoch vs 8
    lr              = 1e-4,
    epochs          = 20,         # model converges by ~15 epochs on this data
    bce_weight      = 1.0,
    physics_reg_wt  = 0.1,
    grad_clip       = 1.0,
    # checkpoint
    ckpt_dir        = 'checkpoints/world_model',
    data_dir        = 'data/rollouts',
)

# GPU config — tuned for 40GB cards (A100-40GB, A6000, etc.)
# Use batch=32 to leave headroom; effective throughput is still fast
if DEVICE == 'cuda':
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_mem_gb >= 70:          # A100-80GB / H100
        CFG.batch_size = 128
    elif gpu_mem_gb >= 35:        # A100-40GB, A6000
        CFG.batch_size = 32
    else:                         # T4 (16GB), V100
        CFG.batch_size = 16
    CFG.base_channels = 64
    CFG.n_train_eps   = 200
    CFG.n_val_eps     = 40
    CFG.num_workers   = 4
else:
    CFG.num_workers   = 0         # MPS/CPU: workers cause issues

CFG.horizon_frames = [int(h * CFG.sim_fps) for h in CFG.prediction_horizons]
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({gpu_mem_gb:.0f} GB)  →  batch_size={CFG.batch_size}')
print('Config:', CFG)

GPU: NVIDIA A100-SXM4-40GB  (42 GB)  →  batch_size=32
Config: namespace(room_size=6.0, camera_height=1.6, camera_fov=90, sim_fps=30, depth_h=128, depth_w=128, num_rays=512, grid_cells=9, steps_per_ep=60, n_train_eps=200, n_val_eps=40, prediction_horizons=[1, 3, 5, 10], base_channels=64, batch_size=32, lr=0.0001, epochs=20, bce_weight=1.0, physics_reg_wt=0.1, grad_clip=1.0, ckpt_dir='checkpoints/world_model', data_dir='data/rollouts', num_workers=4, horizon_frames=[30, 90, 150, 300])


In [3]:
# ── Cell 3: MuJoCo environment ───────────────────────────────────────
try:
    import mujoco
    MUJOCO_OK = True
except ImportError:
    MUJOCO_OK = False
    print('WARNING: mujoco not installed — will use dummy data')

_ROOM_XML = textwrap.dedent("""
<mujoco model="shadowweave">
  <option timestep="0.01" gravity="0 0 -9.81"/>
  <asset>
    <texture name="ft" type="2d" builtin="checker" width="512" height="512"
             rgb1="0.2 0.2 0.2" rgb2="0.3 0.3 0.3"/>
    <material name="fm" texture="ft" texrepeat="6 6"/>
  </asset>
  <worldbody>
    <geom name="floor" type="plane" size="3 3 0.1" material="fm"/>
    <geom name="wall_n" type="box" pos="0 3 1.5"  size="3 0.1 1.5" rgba="0.5 0.5 0.5 1"/>
    <geom name="wall_s" type="box" pos="0 -3 1.5" size="3 0.1 1.5" rgba="0.5 0.5 0.5 1"/>
    <geom name="wall_e" type="box" pos="3 0 1.5"  size="0.1 3 1.5" rgba="0.5 0.5 0.5 1"/>
    <geom name="wall_w" type="box" pos="-3 0 1.5" size="0.1 3 1.5" rgba="0.5 0.5 0.5 1"/>
    <body name="camera_body" pos="0 -2 {camera_height}">
      <camera name="fp_cam" fovy="{fov}" xyaxes="1 0 0 0 0 1"/>
    </body>
    {objects_xml}
  </worldbody>
  {actuators_xml}
</mujoco>
""")


def _static_xml():
    boxes = ''
    for i, (x, y) in enumerate([(1.0, 0.5), (-1.0, 1.0), (0.5, -1.5)]):
        boxes += f'<geom name="obs_{i}" type="box" pos="{x} {y} 0.25" size="0.25 0.25 0.25" rgba="0.8 0.4 0.1 1"/>\n    '
    return boxes, ''


def _moving_xml():
    bodies, acts = '', '<actuator>\n'
    for i in range(3):
        x = i * 1.5 - 1.5
        bodies += f'<body name="mover_{i}" pos="{x} 0 0.25"><freejoint/><geom type="cylinder" size="0.2 0.25" rgba="0.2 0.6 0.9 1"/></body>\n'
    acts += '</actuator>'
    return bodies, acts


def _debris_xml():
    bodies = ''
    for i in range(5):
        x, y = np.random.uniform(-2.5, 2.5), np.random.uniform(-2.5, 2.5)
        z = np.random.uniform(2.0, 4.0)
        bodies += f'<body name="debris_{i}" pos="{x:.2f} {y:.2f} {z:.2f}"><freejoint/><geom type="box" size="0.15 0.15 0.15" rgba="0.9 0.2 0.2 1" mass="1"/></body>\n'
    return bodies, ''


class Env:
    """Thin MuJoCo wrapper — exports depth + occupancy + velocity."""

    def __init__(self, cfg):
        self.cfg = cfg
        self._model = self._data = self._renderer = None

    def reset(self, difficulty='static', seed=None):
        if seed is not None:
            np.random.seed(seed)
        xml_fns = {'static': _static_xml, 'moving': _moving_xml, 'debris': _debris_xml}
        obj_xml, act_xml = xml_fns[difficulty]()
        xml = _ROOM_XML.format(
            camera_height=self.cfg.camera_height, fov=self.cfg.camera_fov,
            objects_xml=obj_xml, actuators_xml=act_xml)
        self._model    = mujoco.MjModel.from_xml_string(xml)
        self._data     = mujoco.MjData(self._model)
        self._renderer = mujoco.Renderer(self._model, height=self.cfg.depth_h, width=self.cfg.depth_w)
        mujoco.mj_resetData(self._model, self._data)
        return self._obs()

    def step(self, n_substeps=3):
        for _ in range(n_substeps):
            mujoco.mj_step(self._model, self._data)
        return self._obs()

    def _obs(self):
        cam = mujoco.mj_name2id(self._model, mujoco.mjtObj.mjOBJ_CAMERA, 'fp_cam')
        self._renderer.update_scene(self._data, camera=cam)

        self._renderer.enable_depth_rendering()
        self._renderer.update_scene(self._data, camera=cam)
        depth_raw = self._renderer.render().astype(np.float32)
        self._renderer.disable_depth_rendering()

        d_min, d_max = depth_raw.min(), depth_raw.max()
        if d_max < 1e-3 or (d_max - d_min) < 1e-3:  # broken rendering: all-zeros or all-ones
            depth_norm = 1.0 - self._occupancy()
        else:
            depth_norm = np.clip(depth_raw / d_max, 0.0, 1.0)

        return {'depth': depth_norm, 'occupancy': self._occupancy(),
                'velocity': self._velocities()}

    def _occupancy(self):
        h, w = self.cfg.depth_h, self.cfg.depth_w
        occ  = np.zeros((h, w), dtype=np.float32)
        room = self.cfg.room_size
        ppm  = w / room
        for i in range(self._model.ngeom):
            if self._model.geom_type[i] == 0:  # plane
                continue
            r_m = float(np.max(self._model.geom_size[i][:2]))
            if r_m > 1.5:           # wall
                continue
            pos = self._data.geom_xpos[i]
            cc  = int((pos[0] + room / 2) / room * w)
            rc  = int((pos[1] + room / 2) / room * h)
            rp  = max(2, int(r_m * ppm))
            for dr in range(-rp, rp + 1):
                for dc in range(-rp, rp + 1):
                    if dr*dr + dc*dc <= rp*rp:
                        r, c = rc + dr, cc + dc
                        if 0 <= r < h and 0 <= c < w:
                            occ[r, c] = 1.0
        mg = max(1, int(0.12 * ppm))
        occ[:mg, :] = occ[-mg:, :] = occ[:, :mg] = occ[:, -mg:] = 1.0
        return occ

    def _velocities(self):
        n = self._model.nbody
        v = np.zeros((n, 3), dtype=np.float32)
        for i in range(n):
            v[i] = self._data.subtree_linvel[i]
        return v

    def close(self):
        if self._renderer:
            self._renderer.close()


if MUJOCO_OK:
    _test_env = Env(CFG)
    _obs = _test_env.reset(difficulty='static', seed=0)
    print('Env OK — depth range:', _obs['depth'].min(), '→', _obs['depth'].max())
    print('  occupancy density:', _obs['occupancy'].mean())
    _test_env.close()
else:
    print('Using dummy data (mujoco not available)')

Env OK — depth range: 0.0 → 1.0
  occupancy density: 0.066467285


In [4]:
# ── Cell 4: ShadowRaycaster (geometric fast path) ────────────────────

class ShadowRaycaster(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_rays  = cfg.num_rays
        self.num_cells = cfg.grid_cells
        self.fov_deg   = cfg.camera_fov
        self.zone_assign = nn.Parameter(torch.randn(self.num_rays, self.num_cells) * 0.1)
        with torch.no_grad():
            for r in range(self.num_rays):
                zi = int(r / self.num_rays * self.num_cells)
                self.zone_assign[r, zi] += 2.0

    @torch.no_grad()
    def forward_from_depth(self, depth_map: torch.Tensor) -> torch.Tensor:
        """(B,1,H,W) → (B,9) uncertainty grid."""
        B = depth_map.shape[0]
        device = depth_map.device
        half_fov = math.radians(self.fov_deg) / 2.0
        azimuths = torch.linspace(-half_fov, half_fov, self.num_rays, device=device)
        norm_x   = torch.tan(azimuths) / math.tan(half_fov)
        norm_y   = torch.zeros_like(norm_x)
        grid = torch.stack([norm_x, norm_y], dim=-1).view(1, 1, self.num_rays, 2).expand(B, -1, -1, -1)
        sampled = F.grid_sample(depth_map, grid, mode='bilinear',
                                padding_mode='border', align_corners=True).squeeze(1).squeeze(1)
        ray_u = (1.0 - sampled).clamp(0.0, 1.0)
        w     = F.softmax(self.zone_assign, dim=0)  # normalise over rays → proper weighted avg
        return torch.matmul(ray_u, w).clamp(0.0, 1.0)


_rc = ShadowRaycaster(CFG)
_d  = torch.rand(2, 1, CFG.depth_h, CFG.depth_w)
_g  = _rc.forward_from_depth(_d)
print('ShadowRaycaster OK — output shape:', _g.shape, '  range:', _g.min().item(), _g.max().item())

ShadowRaycaster OK — output shape: torch.Size([2, 9])   range: 0.46531879901885986 0.5382247567176819


In [5]:
# ── Cell 5: WorldModel (U-Net) ───────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.GELU(),
            nn.Conv2d(oc, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.GELU())
    def forward(self, x): return self.net(x)


class WorldModel(nn.Module):
    """U-Net predicting future occupancy at T horizons.
    Input:  shadow_map (B,1,H,W) + velocity (B,2,H,W)
    Output: (B, T, H, W) raw logits — apply sigmoid for probabilities
    """
    def __init__(self, cfg):
        super().__init__()
        c = cfg.base_channels
        T = len(cfg.prediction_horizons)
        self.enc1 = DoubleConv(3, c)
        self.enc2 = DoubleConv(c, c*2)
        self.enc3 = DoubleConv(c*2, c*4)
        self.enc4 = DoubleConv(c*4, c*8)
        self.bot  = DoubleConv(c*8, c*16)
        self.up4  = nn.ConvTranspose2d(c*16, c*8, 2, stride=2)
        self.dec4 = DoubleConv(c*16, c*8)
        self.up3  = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.dec3 = DoubleConv(c*8, c*4)
        self.up2  = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dec2 = DoubleConv(c*4, c*2)
        self.up1  = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.dec1 = DoubleConv(c*2, c)
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(c, T, 1)

    def forward(self, shadow, vel):
        x  = torch.cat([shadow, vel], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bot(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)  # raw logits — use bce_with_logits for training


_model = WorldModel(CFG)
n_params = sum(p.numel() for p in _model.parameters())
_s = torch.rand(1, 1, CFG.depth_h, CFG.depth_w)
_v = torch.rand(1, 2, CFG.depth_h, CFG.depth_w)
_p = _model(_s, _v)
print(f'WorldModel OK — {n_params/1e6:.1f}M params  output: {_p.shape}')

WorldModel OK — 31.0M params  output: torch.Size([1, 4, 128, 128])


In [6]:
# ── Cell 6: Dataset ──────────────────────────────────────────────────

class RolloutDataset(Dataset):
    def __init__(self, data_dir, split='train'):
        self.files = sorted(pathlib.Path(data_dir, split).glob('*.npz'))
        if not self.files:
            raise FileNotFoundError(f'No .npz files in {data_dir}/{split}')
        self._index = []
        for fi, f in enumerate(self.files):
            n = np.load(f, mmap_mode='r')['shadow_map'].shape[0]
            self._index += [(fi, t) for t in range(n)]
        self._cache = {}

    def __len__(self): return len(self._index)

    def __getitem__(self, idx):
        fi, t = self._index[idx]
        if fi not in self._cache:
            self._cache[fi] = np.load(self.files[fi], mmap_mode='r')
        d = self._cache[fi]
        return {
            'shadow_map':       torch.from_numpy(d['shadow_map'][t]),
            'velocity':         torch.from_numpy(d['velocity'][t]),
            'future_occupancy': torch.from_numpy(d['future_occupancy'][t]),
        }

print('RolloutDataset class ready')

RolloutDataset class ready


In [7]:
# ── Cell 7: Data generation ──────────────────────────────────────────

def gen_velocity_field(depth_t, depth_prev, H, W):
    diff = depth_t - depth_prev
    vx   = np.roll(diff, 1, axis=1) - diff
    vy   = np.roll(diff, 1, axis=0) - diff
    return np.stack([vx, vy], axis=0).astype(np.float32)


def generate_rollouts(cfg, n_eps, data_dir, split, steps_per_ep=60):
    out = pathlib.Path(data_dir, split)
    out.mkdir(parents=True, exist_ok=True)

    horizon_frames = [int(h * cfg.sim_fps) for h in cfg.prediction_horizons]
    max_hf         = max(horizon_frames)
    n_hor          = len(cfg.prediction_horizons)
    H, W           = cfg.depth_h, cfg.depth_w
    difficulties   = ['static', 'moving', 'debris']
    raycaster      = ShadowRaycaster(cfg)
    env            = Env(cfg)

    ep_bar = tqdm(range(n_eps), desc=f'Gen {split}', unit='ep')
    t0 = time.time()
    for ep in ep_bar:
        diff = difficulties[ep % 3]
        env.reset(difficulty=diff, seed=ep)

        n_sim = steps_per_ep + max_hf + 1
        buf = []
        for _ in tqdm(range(n_sim), desc=f'  ep{ep:03d} sim', leave=False, unit='step'):
            buf.append(env.step())

        sm = np.zeros((steps_per_ep, 1, H, W), dtype=np.float32)
        vl = np.zeros((steps_per_ep, 2, H, W), dtype=np.float32)
        fo = np.zeros((steps_per_ep, n_hor, H, W), dtype=np.float32)
        ug = np.zeros((steps_per_ep, 9), dtype=np.float32)

        for t in range(steps_per_ep):
            depth_t = buf[t]['depth']
            sm[t, 0] = depth_t
            if t > 0:
                vl[t] = gen_velocity_field(depth_t, buf[t-1]['depth'], H, W)
            for hi, hf in enumerate(horizon_frames):
                fi = min(t + hf, len(buf) - 1)
                fo[t, hi] = buf[fi]['occupancy']
            with torch.no_grad():
                dt = torch.from_numpy(depth_t).unsqueeze(0).unsqueeze(0)
                ug[t] = raycaster.forward_from_depth(dt)[0].numpy()

        np.savez_compressed(out / f'ep{ep:05d}.npz',
                            shadow_map=sm, velocity=vl,
                            future_occupancy=fo, uncertainty_grid=ug)
        ep_bar.set_postfix(diff=diff, elapsed=f'{time.time()-t0:.0f}s')

    env.close()
    ep_bar.close()
    print(f'Done: {n_eps} ep → {out}  ({time.time()-t0:.1f}s)')


def make_dummy_rollouts(cfg, n_eps, data_dir, split, steps_per_ep=20):
    out = pathlib.Path(data_dir, split)
    out.mkdir(parents=True, exist_ok=True)
    H, W, n_hor = cfg.depth_h, cfg.depth_w, len(cfg.prediction_horizons)
    for ep in tqdm(range(n_eps), desc=f'Dummy {split}', unit='ep'):
        occ = (np.random.rand(H, W) > 0.9).astype(np.float32)
        sm  = np.random.rand(steps_per_ep, 1, H, W).astype(np.float32)
        fo  = np.tile(occ, (steps_per_ep, n_hor, 1, 1)).astype(np.float32)
        vl  = np.zeros((steps_per_ep, 2, H, W), dtype=np.float32)
        ug  = np.random.rand(steps_per_ep, 9).astype(np.float32)
        np.savez_compressed(out / f'ep{ep:05d}.npz',
                            shadow_map=sm, velocity=vl,
                            future_occupancy=fo, uncertainty_grid=ug)
    print(f'Dummy data: {n_eps} ep → {out}')


print('Data generation functions ready')


Data generation functions ready


In [8]:
# ── Cell 8: Generate data ─────────────────────────────────────────────
# Skip if data already exists

train_dir = pathlib.Path(CFG.data_dir, 'train')
val_dir   = pathlib.Path(CFG.data_dir, 'val')

n_existing_train = len(list(train_dir.glob('*.npz'))) if train_dir.exists() else 0
n_existing_val   = len(list(val_dir.glob('*.npz')))   if val_dir.exists()   else 0

print(f'Existing data: {n_existing_train} train / {n_existing_val} val episodes')

NEED_TRAIN = n_existing_train < CFG.n_train_eps
NEED_VAL   = n_existing_val   < CFG.n_val_eps

if NEED_TRAIN or NEED_VAL:
    if MUJOCO_OK:
        if NEED_TRAIN:
            print(f'Generating {CFG.n_train_eps} train episodes...')
            generate_rollouts(CFG, CFG.n_train_eps, CFG.data_dir, 'train')
        if NEED_VAL:
            print(f'Generating {CFG.n_val_eps} val episodes...')
            generate_rollouts(CFG, CFG.n_val_eps, CFG.data_dir, 'val')
    else:
        print('MuJoCo unavailable — generating dummy data')
        make_dummy_rollouts(CFG, CFG.n_train_eps, CFG.data_dir, 'train')
        make_dummy_rollouts(CFG, CFG.n_val_eps,   CFG.data_dir, 'val')
else:
    print('Sufficient data already on disk — skipping generation')

# Verify dataset
train_ds = RolloutDataset(CFG.data_dir, 'train')
val_ds   = RolloutDataset(CFG.data_dir, 'val')
print(f'Dataset: {len(train_ds)} train / {len(val_ds)} val timesteps')
sample = train_ds[0]
for k, v in sample.items():
    print(f'  {k}: {v.shape} {v.dtype}')

Existing data: 0 train / 0 val episodes
Generating 200 train episodes...


Gen train:   0%|          | 0/200 [00:00<?, ?ep/s]

  ep000 sim:   0%|          | 0/361 [00:00<?, ?step/s]

  ep001 sim:   0%|          | 0/361 [00:00<?, ?step/s]

  ep002 sim:   0%|          | 0/361 [00:00<?, ?step/s]

...

  ep036 sim:   0%|          | 0/361 [00:00<?, ?step/s]

  ep037 sim:   0%|          | 0/361 [00:00<?, ?step/s]

  ep038 sim:   0%|          | 0/361 [00:00<?, ?step/s]

  ep039 sim:   0%|          | 0/361 [00:00<?, ?step/s]

Done: 40 ep → data/rollouts/val  (43.1s)
Dataset: 12000 train / 2400 val timesteps
  shadow_map: torch.Size([1, 128, 128]) torch.float32
  velocity: torch.Size([2, 128, 128]) torch.float32
  future_occupancy: torch.Size([4, 128, 128]) torch.float32


In [9]:
# ── Cell 9: Training ──────────────────────────────────────────────────
import gc

# Free any GPU memory from a previous (possibly failed) run of this cell
for _stale in ['model', 'opt', 'sched', 'scaler']:
    if _stale in globals():
        del globals()[_stale]
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def compute_iou(pred: torch.Tensor, target: torch.Tensor, thresh: float = 0.0) -> float:
    # pred is logits; thresh=0.0 ≡ sigmoid > 0.5
    p = (pred.float() > thresh).float()
    t = (target.float() > 0.5).float()
    inter = (p * t).sum().item()
    union = ((p + t) > 0).float().sum().item()
    return inter / max(union, 1.0)


pin       = DEVICE == 'cuda'
n_workers = CFG.num_workers
train_dl  = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                       num_workers=n_workers, pin_memory=pin,
                       persistent_workers=(n_workers > 0))
val_dl    = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False,
                       num_workers=n_workers, pin_memory=pin,
                       persistent_workers=(n_workers > 0))

model = WorldModel(CFG).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.epochs)

ckpt_dir = pathlib.Path(CFG.ckpt_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)

# AMP setup — bf16 on A100 (numerically safe + Tensor Core speed)
#             fp16 on T4 (no bf16 support)
#             disabled on MPS/CPU
if DEVICE == 'cuda':
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    use_amp   = True
else:
    amp_dtype = torch.float16
    use_amp   = False

scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and amp_dtype == torch.float16))

def autocast():
    return torch.amp.autocast(device_type=DEVICE, dtype=amp_dtype, enabled=use_amp)

if DEVICE == 'cuda':
    alloc_gb  = torch.cuda.memory_allocated() / 1e9
    reserv_gb = torch.cuda.memory_reserved() / 1e9
    print(f'GPU after model init: {alloc_gb:.2f} GB allocated / {reserv_gb:.2f} GB reserved')

best_val   = float('inf')
history    = []
patience   = 5
no_improve = 0

n_train_b = len(train_dl)
n_val_b   = len(val_dl)
amp_label  = f'{amp_dtype}'.split('.')[-1] if use_amp else 'fp32'
print(f'Training on {DEVICE} ({amp_label})  |  batch={CFG.batch_size}  |  {n_train_b} train / {n_val_b} val batches/ep')

epoch_bar = tqdm(range(CFG.epochs), desc='Epochs', unit='ep')

for epoch in epoch_bar:
    # ── train ──────────────────────────────────────────────────────
    model.train()
    tr_loss    = 0.0
    train_pbar = tqdm(train_dl, desc='  Train', leave=False, unit='batch', total=n_train_b)

    for batch in train_pbar:
        shadow = batch['shadow_map'].to(DEVICE)
        vel    = batch['velocity'].to(DEVICE)
        target = batch['future_occupancy'].to(DEVICE)

        with autocast():
            pred = model(shadow, vel)
            bce  = F.binary_cross_entropy_with_logits(pred, target)
            phys = (pred[:, 1:] - pred[:, :-1]).abs().mean()
            loss = CFG.bce_weight * bce + CFG.physics_reg_wt * phys

        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt)
        scaler.update()

        tr_loss += loss.item()
        train_pbar.set_postfix(loss=f'{loss.item():.4f}')

    tr_loss /= n_train_b
    sched.step()

    # ── val ────────────────────────────────────────────────────────
    model.eval()
    va_loss, va_iou = 0.0, 0.0
    val_pbar = tqdm(val_dl, desc='  Val  ', leave=False, unit='batch', total=n_val_b)

    with torch.no_grad():
        for batch in val_pbar:
            shadow = batch['shadow_map'].to(DEVICE)
            vel    = batch['velocity'].to(DEVICE)
            target = batch['future_occupancy'].to(DEVICE)

            with autocast():
                pred = model(shadow, vel)

            va_loss += F.binary_cross_entropy_with_logits(pred.float(), target.float()).item()
            va_iou  += compute_iou(pred, target)
            val_pbar.set_postfix(loss=f'{va_loss / (val_pbar.n or 1):.4f}')

    va_loss /= n_val_b
    va_iou  /= n_val_b

    history.append({'epoch': epoch + 1, 'train': tr_loss, 'val': va_loss, 'iou': va_iou})
    epoch_bar.set_postfix(
        train=f'{tr_loss:.4f}', val=f'{va_loss:.4f}', iou=f'{va_iou:.3f}',
        best=f'{best_val:.4f}' if best_val < float('inf') else '—')

    if va_loss < best_val:
        best_val   = va_loss
        no_improve = 0
        torch.save(model.state_dict(), ckpt_dir / 'best.pt')
    else:
        no_improve += 1
        if no_improve >= patience:
            tqdm.write(f'Early stop at epoch {epoch+1} — no val improvement for {patience} epochs')
            break

epoch_bar.close()
torch.save(model.state_dict(), ckpt_dir / 'final.pt')
print(f'Done. Best val loss: {best_val:.4f}  →  {ckpt_dir}/best.pt')

GPU after model init: 0.12 GB allocated / 0.13 GB reserved
Training on cuda (bfloat16)  |  batch=32  |  375 train / 75 val batches/ep


Epochs:   0%|          | 0/20 [00:00<?, ?ep/s]

  Train:   0%|          | 0/375 [00:00<?, ?batch/s]

  Val  :   0%|          | 0/75 [00:00<?, ?batch/s]

  Train:   0%|          | 0/375 [00:00<?, ?batch/s]

BadZipFile: Caught BadZipFile in DataLoader worker process 2.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_12116/3406213331.py", line 23, in __getitem__
    'velocity':         torch.from_numpy(d['velocity'][t]),
                                         ~^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 251, in __getitem__
    bytes = self.zip.open(key)
            ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1676, in open
    raise BadZipFile(
zipfile.BadZipFile: Overlapped entries: 'velocity.n

In [ ]:
# ── Cell 10: Evaluation per horizon ──────────────────────────────────

model.eval()
horizon_names   = [f'{h}s' for h in CFG.prediction_horizons]
n_hor           = len(CFG.prediction_horizons)
iou_per_horizon = np.zeros(n_hor)
n_batches       = 0

with torch.no_grad():
    for batch in tqdm(val_dl, desc='Evaluating', unit='batch'):
        shadow = batch['shadow_map'].to(DEVICE)
        vel    = batch['velocity'].to(DEVICE)
        target = batch['future_occupancy'].to(DEVICE)

        with autocast():
            pred = model(shadow, vel)

        for hi in range(n_hor):
            iou_per_horizon[hi] += compute_iou(pred[:, hi], target[:, hi])
        n_batches += 1

iou_per_horizon /= n_batches

print('\nIOU per prediction horizon (target > 60% at 5s):')
print('─' * 40)
for name, iou in zip(horizon_names, iou_per_horizon):
    bar  = '█' * int(iou * 30)
    flag = '  ✓' if iou > 0.6 else '  (needs more training)'
    print(f'  {name:>4}: {iou:.3f}  {bar}{flag}')
print('─' * 40)
print(f'Mean IOU: {iou_per_horizon.mean():.3f}')


In [ ]:
# ── Cell 11: Training curve ───────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use("Agg")  # non-interactive — plt.show() blocks on macOS
    import matplotlib.pyplot as plt
    epochs  = [h['epoch'] for h in history]
    tr_vals = [h['train'] for h in history]
    va_vals = [h['val']   for h in history]
    ious    = [h['iou']   for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, tr_vals, label='train loss')
    ax1.plot(epochs, va_vals, label='val loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE + physics loss')
    ax1.set_title('Training curve'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, ious, color='green', label='val IOU')
    ax2.axhline(0.6, color='red', linestyle='--', label='target (0.6)')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Mean IOU')
    ax2.set_title('Validation IOU'); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(ckpt_dir) + '/training_curve.png', dpi=120)
    # plt.show() omitted — use Agg backend for headless/script runs
    print('Curve saved to', ckpt_dir / 'training_curve.png')
except ImportError:
    print('matplotlib not available — skipping plot')

In [ ]:
# ── Cell 12: Download checkpoint (Colab) / show path (local) ─────────
best_path  = ckpt_dir / 'best.pt'
final_path = ckpt_dir / 'final.pt'

print(f'Checkpoints saved:')
print(f'  best:  {best_path}  ({best_path.stat().st_size / 1e6:.1f} MB)')
print(f'  final: {final_path} ({final_path.stat().st_size / 1e6:.1f} MB)')

if IN_COLAB:
    from google.colab import files
    print('\nDownloading best.pt ...')
    files.download(str(best_path))
else:
    print(f'\nLocal: checkpoint is at {best_path.resolve()}')
    print('Next step: train_rl.py — PPO local agent (Step 4 in build order)')